# BESS Experiment Charts

Generate the five portfolio charts used to explain the BESS experiment results. The notebook reads the experiment summary CSV, selects an illustrative 48-hour window reproducibly, and saves PNG files to `docs/assets/`.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.battery.data import load_smart_company_analysis, prepare_simulation_data
from src.battery.heuristic import run_heuristic_dispatch
from src.battery.optimization import run_optimized_dispatch
from src.battery.scenarios import (
    make_battery_parameters,
    make_dynamic_surplus_and_grid_charging_scenario,
)

RESULTS_PATH = PROJECT_ROOT / "results" / "bess_experiment_results.csv"
ASSETS_DIR = PROJECT_ROOT / "docs" / "assets"
ASSETS_DIR.mkdir(parents=True, exist_ok=True)

COLORS = {
    "heuristic": "#167D8D",
    "lp_optimization": "#D1495B",
    "demand": "#334E68",
    "surplus": "#2E8B57",
    "price": "#7A5195",
}
METHOD_LABELS = {
    "heuristic": "Heuristic",
    "lp_optimization": "LP optimization",
}
SCENARIO_LABELS = {
    "fixed_surplus_only": "Fixed surplus-only",
    "dynamic_surplus_only": "Dynamic surplus-only",
    "dynamic_surplus_grid_charging": "Dynamic grid-charging",
}

CHART_FONT = "Arial"
TEXT_COLOR = "#172B4D"
MUTED_TEXT_COLOR = "#52616B"
GRID_COLOR = "#E6E9ED"
ZERO_LINE_COLOR = "#AAB2BD"
TITLE_SIZE = 24
AXIS_TITLE_SIZE = 17
TICK_SIZE = 14
LEGEND_SIZE = 14
ANNOTATION_SIZE = 14
DATA_LABEL_SIZE = 14


def save_figure(fig, filename, width=1400, height=800, legend=None, margin=None):
    fig.update_layout(
        template="plotly_white",
        font=dict(family=CHART_FONT, size=TICK_SIZE, color=TEXT_COLOR),
        title=dict(
            x=0.02,
            xanchor="left",
            font=dict(size=TITLE_SIZE, color=TEXT_COLOR),
        ),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            x=0,
            font=dict(size=LEGEND_SIZE),
        ),
        margin=dict(l=80, r=40, t=110, b=70),
        hovermode="x unified",
        uniformtext=dict(mode="show", minsize=DATA_LABEL_SIZE - 2),
    )
    if legend is not None:
        fig.update_layout(legend=legend)
    if margin is not None:
        fig.update_layout(margin=margin)
    fig.update_traces(textfont=dict(size=DATA_LABEL_SIZE))
    fig.update_annotations(font_size=ANNOTATION_SIZE)
    fig.update_xaxes(
        showgrid=False,
        title_font=dict(size=AXIS_TITLE_SIZE),
        tickfont=dict(size=TICK_SIZE),
    )
    fig.update_yaxes(
        gridcolor=GRID_COLOR,
        zerolinecolor=ZERO_LINE_COLOR,
        title_font=dict(size=AXIS_TITLE_SIZE),
        tickfont=dict(size=TICK_SIZE),
    )
    output_path = ASSETS_DIR / filename
    fig.write_image(output_path, width=width, height=height, scale=1)
    print(f"Saved {output_path.relative_to(PROJECT_ROOT)}")
    return fig

results_df = pd.read_csv(RESULTS_PATH)
results_df.shape

## 1. Strategy comparison at 1000 kWh

In [ ]:
strategy_df = results_df[
    (results_df["capacity_kwh"] == 1000)
    & (results_df["method"].isin(METHOD_LABELS))
].copy()
scenario_order = list(SCENARIO_LABELS)

strategy_fig = go.Figure()
for method in METHOD_LABELS:
    method_df = (
        strategy_df[strategy_df["method"] == method]
        .set_index("scenario")
        .reindex(scenario_order)
    )
    values = method_df["cost_savings_eur"]
    strategy_fig.add_bar(
        name=METHOD_LABELS[method],
        x=[SCENARIO_LABELS[name] for name in scenario_order],
        y=values,
        marker_color=COLORS[method],
        text=[f"{value / 1000:.1f}k EUR" for value in values],
        textposition="outside",
    )

strategy_fig.update_layout(
    title="Annual Operational Savings by Dispatch Strategy - 1000 kWh BESS",
    barmode="group",
    yaxis_title="Annual operational savings (EUR)",
)
strategy_fig.add_annotation(
    text="Savings exclude BESS investment cost and use the corresponding fixed or dynamic no-BESS baseline.",
    xref="paper", yref="paper", x=0, y=-0.18, showarrow=False,
    font=dict(color=MUTED_TEXT_COLOR),
)
strategy_fig = save_figure(strategy_fig, "strategy_comparison_1000kwh.png")
strategy_fig

## 2. Arbitrage value at 1000 kWh

Show how the average discharge-hour value separates into grid-charge price, efficiency and degradation cost, and remaining net spread.


In [ ]:
arbitrage_df = (
    results_df[
        (results_df["capacity_kwh"] == 1000)
        & (results_df["scenario"] == "dynamic_surplus_grid_charging")
        & (results_df["method"].isin(METHOD_LABELS))
    ]
    .set_index("method")
    .reindex(METHOD_LABELS)
)

arbitrage_df["efficiency_degradation_cost_eur_per_kwh"] = (
    arbitrage_df["average_grid_charge_price_eur_per_kwh"]
    / (arbitrage_df["eta_charge"] * arbitrage_df["eta_discharge"])
    + arbitrage_df["degradation_cost_eur_per_kwh"]
    - arbitrage_df["average_grid_charge_price_eur_per_kwh"]
)
arbitrage_df["discharge_value_eur_per_kwh"] = (
    arbitrage_df["average_grid_charge_price_eur_per_kwh"]
    + arbitrage_df["efficiency_degradation_cost_eur_per_kwh"]
    + arbitrage_df["grid_charge_arbitrage_spread_eur_per_kwh"]
)
ordered_methods = list(METHOD_LABELS)
method_labels = [METHOD_LABELS[method] for method in ordered_methods]

arbitrage_fig = go.Figure()
arbitrage_fig.add_bar(
    name="Grid-charge price",
    x=method_labels,
    y=arbitrage_df.loc[ordered_methods, "average_grid_charge_price_eur_per_kwh"],
    marker_color="#8FA1B3",
    text=[
        f"{value * 100:.1f} ct"
        for value in arbitrage_df.loc[ordered_methods, "average_grid_charge_price_eur_per_kwh"]
    ],
    textposition="inside",
    insidetextanchor="middle",
)
arbitrage_fig.add_bar(
    name="Efficiency + degradation",
    x=method_labels,
    y=arbitrage_df.loc[ordered_methods, "efficiency_degradation_cost_eur_per_kwh"],
    marker_color="#D8DEE6",
    text=[
        f"{value * 100:.1f} ct"
        for value in arbitrage_df.loc[ordered_methods, "efficiency_degradation_cost_eur_per_kwh"]
    ],
    textposition="inside",
    insidetextanchor="middle",
)
arbitrage_fig.add_bar(
    name="Net spread",
    x=method_labels,
    y=arbitrage_df.loc[ordered_methods, "grid_charge_arbitrage_spread_eur_per_kwh"],
    marker_color=COLORS["heuristic"],
    text=[
        f"+{value * 100:.1f} ct"
        for value in arbitrage_df.loc[ordered_methods, "grid_charge_arbitrage_spread_eur_per_kwh"]
    ],
    textposition="inside",
    insidetextanchor="middle",
)

for method in ordered_methods:
    row = arbitrage_df.loc[method]
    arbitrage_fig.add_annotation(
        x=METHOD_LABELS[method],
        y=row["discharge_value_eur_per_kwh"] + 0.008,
            text=f"{row['discharge_value_eur_per_kwh'] * 100:.1f} ct/kWh",
        showarrow=False,
        yanchor="bottom",
        font=dict(color=MUTED_TEXT_COLOR),
    )

x_max = float(arbitrage_df.loc[ordered_methods, "discharge_value_eur_per_kwh"].max())
arbitrage_fig.update_layout(
    title="Grid-Charging Arbitrage Value by Dispatch Method",
    barmode="stack",
    bargap=0.28,
)
arbitrage_fig.update_xaxes(title_text="")
arbitrage_fig.update_yaxes(
    title_text="Average discharge-hour value (ct/kWh)",
    tickvals=[0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30],
    ticktext=["0", "5", "10", "15", "20", "25", "30"],
    range=[0, x_max + 0.055],
)
arbitrage_fig = save_figure(
    arbitrage_fig,
    "arbitrage_value_1000kwh.png",
    width=950,
    height=780,
    margin=dict(l=85, r=45, t=105, b=85),
)
arbitrage_fig


## 3. Capacity sensitivity

In [ ]:
capacity_df = results_df[
    (results_df["scenario"] == "dynamic_surplus_grid_charging")
    & (results_df["method"].isin(METHOD_LABELS))
].sort_values(["method", "capacity_kwh"])

capacity_fig = go.Figure()
for method in METHOD_LABELS:
    method_df = capacity_df[capacity_df["method"] == method]
    capacity_fig.add_scatter(
        name=METHOD_LABELS[method],
        x=method_df["capacity_kwh"],
        y=method_df["cost_savings_eur"],
        mode="lines+markers+text",
        line=dict(color=COLORS[method], width=3),
        marker=dict(size=10),
        text=[f"{value / 1000:.1f}k" for value in method_df["cost_savings_eur"]],
        textposition="top center",
    )

capacity_fig.update_layout(
    title="BESS Capacity Sensitivity - Dynamic Grid-Charging Scenario",
    xaxis_title="Battery capacity (kWh)",
    yaxis_title="Annual operational savings (EUR)",
)
capacity_fig = save_figure(capacity_fig, "capacity_sensitivity.png")
capacity_fig

## 4. Capacity utilization

In [ ]:
utilization_df = capacity_df[capacity_df["method"] == "lp_optimization"].copy()

utilization_fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.16,
    subplot_titles=("Surplus captured", "Equivalent full cycles per year"),
)
utilization_fig.add_scatter(
    x=utilization_df["capacity_kwh"],
    y=utilization_df["surplus_capture_ratio"] * 100,
    mode="lines+markers+text",
    line=dict(color=COLORS["surplus"], width=3),
    marker=dict(size=10),
    text=[f"{value:.1f}%" for value in utilization_df["surplus_capture_ratio"] * 100],
    textposition="top center",
    name="Surplus capture",
    row=1, col=1,
)
utilization_fig.add_scatter(
    x=utilization_df["capacity_kwh"],
    y=utilization_df["approximate_cycles"],
    mode="lines+markers+text",
    line=dict(color=COLORS["price"], width=3),
    marker=dict(size=10),
    text=[f"{value:.0f}" for value in utilization_df["approximate_cycles"]],
    textposition="top center",
    name="Equivalent cycles",
    row=2, col=1,
)
utilization_fig.update_yaxes(title_text="Surplus capture (%)", row=1, col=1)
utilization_fig.update_yaxes(title_text="Cycles per year", row=2, col=1)
utilization_fig.update_xaxes(title_text="Battery capacity (kWh)", row=2, col=1)
utilization_fig.update_layout(
    title="Larger Batteries Capture More Surplus but Cycle Less",
    showlegend=False,
)
utilization_fig = save_figure(utilization_fig, "capacity_utilization.png", height=950)
utilization_fig

## 5. Reproducible 48-hour dispatch rollout

The selected window must contain surplus, demand, grid charging, and battery discharge for both methods. Candidate windows are ranked by their 48-hour price range. The simulation includes 24 additional hours after the displayed window so every displayed LP step retains its full planning horizon.

In [ ]:
battery = make_battery_parameters(
    capacity_kwh=1000,
    c_rate=0.5,
    min_soc_fraction=0.10,
    max_soc_fraction=1.00,
    eta_charge=0.95,
    eta_discharge=0.95,
    degradation_cost_eur_per_kwh=0.03,
)
scenario = make_dynamic_surplus_and_grid_charging_scenario(
    export_price_eur_per_kwh=0.08,
    import_markup_eur_per_kwh=0.115,
    horizon_hours=24,
    surplus_reserve_fraction=1.0,
    grid_connection_limit_kw=500.0,
)
analysis_df = (
    load_smart_company_analysis()
    .sort_values("local_timestamp")
    .reset_index(drop=True)
)
prepared_df = prepare_simulation_data(analysis_df, scenario)

DISPLAY_HOURS = 48
SIMULATION_HOURS = DISPLAY_HOURS + scenario.horizon_hours
candidate_windows = []
for start in range(len(prepared_df) - SIMULATION_HOURS + 1):
    window = prepared_df.iloc[start : start + DISPLAY_HOURS]
    surplus_kwh = float(window["available_surplus_kwh"].sum())
    deficit_kwh = float(window["demand_after_generation_kwh"].sum())
    price_range = float(
        window["dynamic_import_price_eur_per_kwh"].max()
        - window["dynamic_import_price_eur_per_kwh"].min()
    )
    if surplus_kwh > 50 and deficit_kwh > 1000:
        candidate_windows.append((price_range, surplus_kwh, start))

selected = None
for price_range, surplus_kwh, start in sorted(candidate_windows, reverse=True):
    simulation_df = analysis_df.iloc[start : start + SIMULATION_HOURS].reset_index(drop=True)
    heuristic_dispatch = run_heuristic_dispatch(simulation_df, battery, scenario)
    lp_dispatch = run_optimized_dispatch(simulation_df, battery, scenario)
    displayed_heuristic = heuristic_dispatch.iloc[:DISPLAY_HOURS].copy()
    displayed_lp = lp_dispatch.iloc[:DISPLAY_HOURS].copy()
    both_methods_active = all(
        dispatch["charge_from_grid_kwh"].sum() > 1
        and dispatch["discharge_to_load_kwh"].sum() > 1
        for dispatch in (displayed_heuristic, displayed_lp)
    )
    if both_methods_active:
        selected = (start, displayed_heuristic, displayed_lp)
        break

if selected is None:
    raise RuntimeError("Could not find a suitable 48-hour comparison window.")

start, heuristic_48h, lp_48h = selected
context_48h = prepared_df.iloc[start : start + DISPLAY_HOURS].reset_index(drop=True)
timestamps = context_48h["local_timestamp"]
print(
    f"Selected window: {timestamps.iloc[0]} to {timestamps.iloc[-1]} | "
    f"surplus={context_48h['available_surplus_kwh'].sum():.0f} kWh | "
    f"price range={context_48h['dynamic_import_price_eur_per_kwh'].max() - context_48h['dynamic_import_price_eur_per_kwh'].min():.3f} EUR/kWh"
)

In [ ]:
rollout_fig = make_subplots(
    rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.07,
    subplot_titles=(
        "Site demand and local surplus",
        "Dynamic import price and heuristic thresholds",
        "Battery dispatch: positive = discharge, negative = charge",
        "Battery state of charge",
    ),
)
rollout_fig.add_scatter(
    x=timestamps, y=context_48h["demand_after_generation_kwh"],
    name="Demand after generation", line=dict(color=COLORS["demand"], width=2),
    fill="tozeroy", row=1, col=1,
)
rollout_fig.add_scatter(
    x=timestamps, y=-context_48h["available_surplus_kwh"],
    name="Available surplus", line=dict(color=COLORS["surplus"], width=2),
    fill="tozeroy", row=1, col=1,
)
rollout_fig.add_scatter(
    x=timestamps, y=context_48h["dynamic_import_price_eur_per_kwh"],
    name="Import price", line=dict(color=COLORS["price"], width=3), row=2, col=1,
)
rollout_fig.add_scatter(
    x=timestamps, y=heuristic_48h["low_price_threshold_eur_per_kwh"],
    name="Heuristic low threshold",
    line=dict(color="#4C956C", width=1.5, dash="dot"), row=2, col=1,
)
rollout_fig.add_scatter(
    x=timestamps, y=heuristic_48h["high_price_threshold_eur_per_kwh"],
    name="Heuristic high threshold",
    line=dict(color="#C44536", width=1.5, dash="dot"), row=2, col=1,
)
for dispatch, method in ((heuristic_48h, "heuristic"), (lp_48h, "lp_optimization")):
    net_dispatch = dispatch["discharge_to_load_kwh"] - dispatch["battery_charge_kwh"]
    rollout_fig.add_scatter(
        x=timestamps, y=net_dispatch, name=METHOD_LABELS[method],
        mode="lines", line=dict(color=COLORS[method], width=2.5, shape="hv"),
        row=3, col=1,
    )
    soc_percent = (
        (dispatch["soc_end_kwh"] - battery.min_soc_kwh)
        / (battery.max_soc_kwh - battery.min_soc_kwh)
        * 100
    )
    rollout_fig.add_scatter(
        x=timestamps, y=soc_percent, name=f"{METHOD_LABELS[method]} SOC",
        mode="lines", line=dict(color=COLORS[method], width=2.5, shape="hv"),
        showlegend=False,
        row=4, col=1,
    )

rollout_fig.update_yaxes(title_text="Energy (kWh)", row=1, col=1)
rollout_fig.update_yaxes(title_text="EUR/kWh", row=2, col=1)
rollout_fig.update_yaxes(title_text="Energy (kWh)", row=3, col=1)
rollout_fig.update_yaxes(title_text="Usable SOC (%)", range=[-3, 103], row=4, col=1)
rollout_fig.update_xaxes(title_text="Local time", row=4, col=1)
rollout_fig.update_layout(
    title=(
        "48-Hour Dispatch Rollout - 1000 kWh Dynamic Grid-Charging BESS"
        f"<br><sup>{timestamps.iloc[0]:%Y-%m-%d %H:%M} to {timestamps.iloc[-1]:%Y-%m-%d %H:%M}</sup>"
    ),
)
rollout_fig = save_figure(
    rollout_fig,
    "dispatch_rollout_48h.png",
    width=1600,
    height=1250,
    legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.01),
    margin=dict(l=80, r=280, t=140, b=70),
)
rollout_fig